# Tuned benchmark: 10 datasets × selected classifiers

In [1]:
from pathlib import Path
import os
import sys
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

In [2]:
def find_repo_root(start_path: Path = Path.cwd()) -> Path:
    current = start_path.resolve()
    while current != current.parent:
        if (current / "scripts" / "run_experiment.py").exists():
            return current
        current = current.parent
    raise FileNotFoundError(
        "Could not find repository root containing scripts/run_experiment.py"
    )

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

print("Working directory:", Path.cwd())
assert Path("scripts/run_experiment.py").exists()
assert Path("scripts/tune_per_configuration.py").exists()
assert Path("scripts/run_best_hyperparameters.py").exists()

Working directory: /Users/gabbo/Documents/GitHub/Thesis_SlidingWindow


In [3]:
# Use the same Python executable as the current Jupyter kernel.
PYTHON_EXECUTABLE = sys.executable
print("Python executable:", PYTHON_EXECUTABLE)

Python executable: /Users/gabbo/Documents/GitHub/Thesis_SlidingWindow/.venv/bin/python


In [4]:
# Check that the local package can be imported from the notebook kernel.
import sliding_window_tsc
print("sliding_window_tsc import successful")

sliding_window_tsc import successful


In [5]:
def run_command(command: list[str], continue_on_error: bool = True) -> int:
    command = command.copy()

    if command[0] == "python":
        command[0] = PYTHON_EXECUTABLE
        command.insert(1, "-u")

    print()
    print("=" * 100)
    print("Running command:")
    print(" ".join(command))
    print("=" * 100)
    print()

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    src_path = str((Path.cwd() / "src").resolve())
    existing_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = src_path + (os.pathsep + existing_pythonpath if existing_pythonpath else "")

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    process.wait()

    if process.returncode != 0 and not continue_on_error:
        raise RuntimeError(f"Command failed with exit code {process.returncode}")

    if process.returncode != 0:
        print(f"WARNING: command failed with exit code {process.returncode}", flush=True)

    return process.returncode

## Benchmark configuration

This uses the same selected classifiers as the current untuned benchmark:

- `MiniRocketClassifier`
- `HydraClassifier`
- `QUANTClassifier`
- `WEASEL`
- `TimeSeriesForestClassifier`
- `Catch22Classifier`
- `RDSTClassifier`
- `KNeighborsTimeSeriesClassifier`

In [6]:
DATASETS = [
    "GunPoint",
    "ECG200",
    "ItalyPowerDemand",
    "Coffee",
    "Fish",
    "Beef",
    "Plane",
    "TwoLeadECG",
    "Wafer",
    "FaceFour",
]

CLASSIFIERS = [
    "MiniRocketClassifier",
    "HydraClassifier",
    "QUANTClassifier",
    "WEASEL",
    "TimeSeriesForestClassifier",
    "Catch22Classifier",
    "RDSTClassifier",
    "KNeighborsTimeSeriesClassifier",
]

WINDOW_START = "0.10"
WINDOW_END = "0.50"
WINDOW_STEP = "0.03"
STRIDE_RATIOS = ["0.1", "0.2", "0.5"]

METRIC = "series_macro_f1"
N_TRIALS = "10"
RANDOM_STATE = "42"

# Keep True for a first check.
SMOKE_TEST = False

if SMOKE_TEST:
    DATASETS_TO_RUN = DATASETS[:1]
    CLASSIFIERS_TO_RUN = CLASSIFIERS[:2]
    WINDOW_START = "0.10"
    WINDOW_END = "0.12"
    WINDOW_STEP = "0.01"
    STRIDE_RATIOS = ["0.5"]
    N_TRIALS = "2"
else:
    DATASETS_TO_RUN = DATASETS
    CLASSIFIERS_TO_RUN = CLASSIFIERS

TUNING_ROOT = Path("results/benchmark_10datasets_tuning/tuned/tuning")
TUNED_ROOT = Path("results/benchmark_10datasets_tuning/tuned/final")
PLOTS_ROOT = Path("plots/benchmark_10datasets_tuning/tuned")

TUNING_ROOT.mkdir(parents=True, exist_ok=True)
TUNED_ROOT.mkdir(parents=True, exist_ok=True)
PLOTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Datasets to run:", DATASETS_TO_RUN)
print("Classifiers to run:", CLASSIFIERS_TO_RUN)
print("Window range:", WINDOW_START, WINDOW_END, WINDOW_STEP)
print("Stride ratios:", STRIDE_RATIOS)
print("N_TRIALS:", N_TRIALS)

Datasets to run: ['GunPoint', 'ECG200', 'ItalyPowerDemand', 'Coffee', 'Fish', 'Beef', 'Plane', 'TwoLeadECG', 'Wafer', 'FaceFour']
Classifiers to run: ['MiniRocketClassifier', 'HydraClassifier', 'QUANTClassifier', 'WEASEL', 'TimeSeriesForestClassifier', 'Catch22Classifier', 'RDSTClassifier', 'KNeighborsTimeSeriesClassifier']
Window range: 0.10 0.50 0.03
Stride ratios: ['0.1', '0.2', '0.5']
N_TRIALS: 10


In [7]:
# Dataset availability check
missing_datasets = []
available_datasets = []

for dataset in DATASETS_TO_RUN:
    dataset_folder = Path("data/raw") / dataset
    if dataset_folder.exists():
        available_datasets.append(dataset)
    else:
        missing_datasets.append(dataset)

print("Available datasets:", available_datasets)
print("Missing datasets:", missing_datasets)

Available datasets: ['GunPoint', 'ECG200', 'ItalyPowerDemand', 'Coffee', 'Fish', 'Beef', 'Plane', 'TwoLeadECG', 'Wafer', 'FaceFour']
Missing datasets: []


In [8]:
# Classifier import check
IMPORT_CHECKS = {
    "MiniRocketClassifier": "from aeon.classification.convolution_based import MiniRocketClassifier",
    "HydraClassifier": "from aeon.classification.convolution_based import HydraClassifier",
    "QUANTClassifier": "from aeon.classification.interval_based import QUANTClassifier",
    "WEASEL": "from aeon.classification.dictionary_based import WEASEL",
    "TimeSeriesForestClassifier": "from aeon.classification.interval_based import TimeSeriesForestClassifier",
    "Catch22Classifier": "from aeon.classification.feature_based import Catch22Classifier",
    "RDSTClassifier": "from aeon.classification.shapelet_based import RDSTClassifier",
    "KNeighborsTimeSeriesClassifier": "from aeon.classification.distance_based import KNeighborsTimeSeriesClassifier",
}

available_classifiers = []

for classifier in CLASSIFIERS_TO_RUN:
    print()
    print(classifier)
    try:
        exec(IMPORT_CHECKS[classifier])
        print("  import successful")
        available_classifiers.append(classifier)
    except Exception as exc:
        print("  import failed:", repr(exc))

CLASSIFIERS_TO_RUN = available_classifiers
print("Classifiers to run:", CLASSIFIERS_TO_RUN)


MiniRocketClassifier
  import successful

HydraClassifier
  import successful

QUANTClassifier
  import successful

WEASEL
  import successful

TimeSeriesForestClassifier
  import successful

Catch22Classifier
  import successful

RDSTClassifier
  import successful

KNeighborsTimeSeriesClassifier
  import successful
Classifiers to run: ['MiniRocketClassifier', 'HydraClassifier', 'QUANTClassifier', 'WEASEL', 'TimeSeriesForestClassifier', 'Catch22Classifier', 'RDSTClassifier', 'KNeighborsTimeSeriesClassifier']


## Step 1 — Tune hyperparameters per configuration

This creates one best-hyperparameter JSON for each `dataset + classifier + window_size + stride_ratio`.

In [9]:
RUN_TUNING = True

if RUN_TUNING:
    tuning_log = []

    for dataset in DATASETS_TO_RUN:
        dataset_folder = Path("data/raw") / dataset

        if not dataset_folder.exists():
            print(f"Skipping missing dataset: {dataset}")
            tuning_log.append({"dataset": dataset, "classifier": None, "status": "missing_dataset", "return_code": None})
            continue

        for classifier in CLASSIFIERS_TO_RUN:
            output_dir = TUNING_ROOT / dataset / classifier / "per_config"
            best_dir = output_dir / "best_hyperparameters"
            index_file = output_dir / "best_hyperparameters_index.csv"

            if index_file.exists() and best_dir.exists() and any(best_dir.glob("*.json")):
                print()
                print(f"Skipping existing tuning: dataset={dataset}, classifier={classifier}")
                print(f"Existing index file: {index_file}")
                tuning_log.append({"dataset": dataset, "classifier": classifier, "status": "already_tuned", "return_code": 0})
                continue

            command = [
                "python", "scripts/tune_per_configuration.py",
                "--dataset-folder", str(dataset_folder),
                "--classifier", classifier,
                "--window-sizes", WINDOW_START, WINDOW_END, WINDOW_STEP,
                "--percentages",
                "--stride-ratios", *STRIDE_RATIOS,
                "--metric", METRIC,
                "--n-trials", N_TRIALS,
                "--output-dir", str(output_dir),
                "--random-state", RANDOM_STATE,
            ]

            return_code = run_command(command, continue_on_error=True)
            tuning_log.append({"dataset": dataset, "classifier": classifier, "status": "executed" if return_code == 0 else "failed_command", "return_code": return_code})

    tuning_log_df = pd.DataFrame(tuning_log)
    display(tuning_log_df)
    tuning_log_df.to_csv(TUNING_ROOT / "tuning_log.csv", index=False)
else:
    print("RUN_TUNING is False. No tuning was launched.")


Skipping existing tuning: dataset=GunPoint, classifier=MiniRocketClassifier
Existing index file: results/benchmark_10datasets_tuning/tuned/tuning/GunPoint/MiniRocketClassifier/per_config/best_hyperparameters_index.csv

Running command:
/Users/gabbo/Documents/GitHub/Thesis_SlidingWindow/.venv/bin/python -u scripts/tune_per_configuration.py --dataset-folder data/raw/GunPoint --classifier HydraClassifier --window-sizes 0.10 0.50 0.03 --percentages --stride-ratios 0.1 0.2 0.5 --metric series_macro_f1 --n-trials 10 --output-dir results/benchmark_10datasets_tuning/tuned/tuning/GunPoint/HydraClassifier/per_config --random-state 42


[1/42] Tuning GunPoint / HydraClassifier / window_size=15 / stride_ratio=0.1
[I 2026-07-14 21:42:14,690] A new study created in memory with name: no-name-2eb6c906-5cae-45e4-ab10-a383e10fdd7a

GunPoint / HydraClassifier: 100%|██████████| 1/1 [00:03<00:00,  3.41s/config, status=ok, stride=1, stride_ratio=0.1, window_size=15]
[I 2026-07-14 21:42:18,111] Trial 0 fini

KeyboardInterrupt: 

## Step 2 — Run experiments with the best hyperparameters

This reruns each tuned configuration using the best hyperparameters saved during Step 1.

In [ ]:
RUN_BEST_HYPERPARAMETERS = True

if RUN_BEST_HYPERPARAMETERS:
    run_best_log = []

    for dataset in DATASETS_TO_RUN:
        for classifier in CLASSIFIERS_TO_RUN:
            best_dir = TUNING_ROOT / dataset / classifier / "per_config" / "best_hyperparameters"
            output_dir = TUNED_ROOT / dataset / classifier

            if not best_dir.exists():
                print(f"Skipping missing best_dir: {best_dir}")
                run_best_log.append({"dataset": dataset, "classifier": classifier, "status": "missing_best_dir", "return_code": None})
                continue

            existing_csv_files = sorted(output_dir.glob("*.csv"))

            if existing_csv_files:
                print()
                print(f"Skipping existing tuned run: dataset={dataset}, classifier={classifier}")
                print(f"Existing file: {existing_csv_files[-1]}")
                run_best_log.append({"dataset": dataset, "classifier": classifier, "status": "already_done", "return_code": 0})
                continue

            output_dir.mkdir(parents=True, exist_ok=True)

            command = [
                "python", "scripts/run_best_hyperparameters.py",
                "--best-dir", str(best_dir),
                "--data-root", "data/raw",
                "--output-dir", str(output_dir),
                "--random-state", RANDOM_STATE,
                "--skip-failed-best",
            ]

            return_code = run_command(command, continue_on_error=True)
            run_best_log.append({"dataset": dataset, "classifier": classifier, "status": "executed" if return_code == 0 else "failed_command", "return_code": return_code})

    run_best_log_df = pd.DataFrame(run_best_log)
    display(run_best_log_df)
    run_best_log_df.to_csv(TUNED_ROOT / "run_best_log.csv", index=False)
else:
    print("RUN_BEST_HYPERPARAMETERS is False. No tuned final runs were launched.")

## Step 3 — Load and combine tuned results

In [ ]:
def load_all_results(root: Path) -> pd.DataFrame:
    rows = []
    csv_files = sorted(root.glob("*/*/*.csv"))
    print("CSV files found:", len(csv_files))

    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
        except Exception as exc:
            print("Could not read:", csv_file, repr(exc))
            continue
        df["source_file"] = str(csv_file)
        rows.append(df)

    if not rows:
        raise FileNotFoundError(f"No CSV files found under {root}")

    return pd.concat(rows, ignore_index=True)

tuned_results = load_all_results(TUNED_ROOT)
print("Combined tuned shape:", tuned_results.shape)
display(tuned_results.head())

In [ ]:
numeric_columns = [
    "window_size", "window_percentage", "stride", "stride_ratio",
    "series_macro_f1", "series_accuracy", "series_balanced_accuracy",
    "fit_time_sec", "predict_time_sec", "total_time_sec",
    "n_train_windows", "n_test_windows",
    "tuning_best_value", "tuning_best_trial",
]

for column in numeric_columns:
    if column in tuned_results.columns:
        tuned_results[column] = pd.to_numeric(tuned_results[column], errors="coerce")

if "status" in tuned_results.columns:
    ok_tuned_results = tuned_results[tuned_results["status"] == "ok"].copy()
else:
    ok_tuned_results = tuned_results.copy()
    ok_tuned_results["status"] = "ok"

print("All tuned rows:", len(tuned_results))
print("OK tuned rows:", len(ok_tuned_results))

if "status" in tuned_results.columns:
    display(tuned_results["status"].value_counts(dropna=False))

combined_file = TUNED_ROOT / "combined_tuned_results.csv"
tuned_results.to_csv(combined_file, index=False)
print("Saved combined tuned results to:", combined_file)

## Plot 1 — Grid plot: rows = classifiers, columns = datasets

Each subplot shows tuned Macro F1 versus window size. The three lines correspond to stride ratios.

In [ ]:
GRID_PLOTS_DIR = PLOTS_ROOT / "macro_f1_grid"
GRID_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

plot_df = ok_tuned_results.copy()
plot_df["window_percentage_100"] = (plot_df["window_percentage"] * 100).round(0).astype(int)

datasets_to_plot = [d for d in DATASETS if d in plot_df["dataset"].unique()]
classifiers_to_plot = [c for c in CLASSIFIERS if c in plot_df["classifier"].unique()]

print("Datasets to plot:", datasets_to_plot)
print("Classifiers to plot:", classifiers_to_plot)

grid_df = (
    plot_df
    .groupby(["dataset", "classifier", "stride_ratio", "window_percentage_100"], as_index=False)
    .agg(series_macro_f1=("series_macro_f1", "mean"), n_rows=("series_macro_f1", "count"))
    .sort_values(["classifier", "dataset", "stride_ratio", "window_percentage_100"])
)

n_rows = len(classifiers_to_plot)
n_cols = len(datasets_to_plot)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(3.2 * max(n_cols, 1), 2.4 * max(n_rows, 1)),
    sharex=True,
    sharey=True,
    facecolor="white",
)

if n_rows == 1 and n_cols == 1:
    axes = np.array([[axes]])
elif n_rows == 1:
    axes = axes.reshape(1, -1)
elif n_cols == 1:
    axes = axes.reshape(-1, 1)

for row_idx, classifier in enumerate(classifiers_to_plot):
    for col_idx, dataset in enumerate(datasets_to_plot):
        ax = axes[row_idx, col_idx]
        subset = grid_df[(grid_df["classifier"] == classifier) & (grid_df["dataset"] == dataset)]

        if subset.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes, fontsize=8)
            ax.set_axis_off()
            continue

        for stride_ratio, group in subset.groupby("stride_ratio"):
            group = group.sort_values("window_percentage_100")
            ax.plot(group["window_percentage_100"], group["series_macro_f1"], linewidth=1.3, label=f"s={stride_ratio}")

        if row_idx == 0:
            ax.set_title(dataset, fontsize=9)
        if col_idx == 0:
            ax.set_ylabel(classifier, fontsize=8)
        ax.grid(True, alpha=0.25)
        ax.tick_params(axis="both", labelsize=7)

fig.supxlabel("Window size (%)", fontsize=11)
fig.supylabel("Series Macro F1", fontsize=11)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, title="Stride ratio", loc="upper center", ncol=len(labels), bbox_to_anchor=(0.5, 1.01))
fig.suptitle("Tuned benchmark - Macro F1 by classifier, dataset and stride ratio", fontsize=14, y=1.035)
fig.tight_layout()

output_file = GRID_PLOTS_DIR / "tuned_macro_f1_grid_classifiers_by_datasets.png"
fig.savefig(output_file, dpi=300, bbox_inches="tight", facecolor="white")
print("Saved:", output_file)
plt.show()

## Plot 2 — Top sliding-window values

Counts how often each window value appears in Top-1, Top-3, and Top-5 positions by tuned Macro F1. Ranking group: `dataset + classifier + stride_ratio`.

In [ ]:
TOP_WINDOW_DIR = PLOTS_ROOT / "top_window_values"
TOP_WINDOW_DIR.mkdir(parents=True, exist_ok=True)

plot_df = ok_tuned_results.copy()
plot_df["window_percentage_100"] = (plot_df["window_percentage"] * 100).round(0).astype(int)

config_df = (
    plot_df
    .groupby(["dataset", "classifier", "stride_ratio", "window_percentage_100"], as_index=False)
    .agg(series_macro_f1=("series_macro_f1", "mean"), n_rows=("series_macro_f1", "count"))
)

ranking_keys = ["dataset", "classifier", "stride_ratio"]

def get_top_k_counts(df: pd.DataFrame, k: int):
    top_rows = []
    for _, group in df.groupby(ranking_keys):
        group = group.sort_values(["series_macro_f1", "window_percentage_100"], ascending=[False, True])
        top_k = group.head(k).copy()
        top_k["top_k"] = k
        top_rows.append(top_k)
    top_k_df = pd.concat(top_rows, ignore_index=True)
    counts = (
        top_k_df
        .groupby("window_percentage_100", as_index=False)
        .agg(count=("window_percentage_100", "size"), mean_macro_f1=("series_macro_f1", "mean"))
        .sort_values("count", ascending=False)
    )
    counts["window_label"] = counts["window_percentage_100"].astype(str) + "%"
    return counts, top_k_df

top1_counts, top1_rows = get_top_k_counts(config_df, k=1)
top3_counts, top3_rows = get_top_k_counts(config_df, k=3)
top5_counts, top5_rows = get_top_k_counts(config_df, k=5)

top1_counts.to_csv(TOP_WINDOW_DIR / "tuned_top1_window_counts.csv", index=False)
top3_counts.to_csv(TOP_WINDOW_DIR / "tuned_top3_window_counts.csv", index=False)
top5_counts.to_csv(TOP_WINDOW_DIR / "tuned_top5_window_counts.csv", index=False)

print("Number of ranking groups:", config_df.groupby(ranking_keys).ngroups)
display(top1_counts.head(10))
display(top3_counts.head(10))
display(top5_counts.head(10))

In [ ]:
plot_specs = [
    {
        "counts": top1_counts,
        "title": "Top-1",
        "subtitle": "Best tuned Macro F1",
    },
    {
        "counts": top3_counts,
        "title": "Top-3",
        "subtitle": "Among best 3 tuned Macro F1",
    },
    {
        "counts": top5_counts,
        "title": "Top-5",
        "subtitle": "Among best 5 tuned Macro F1",
    },
]

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 6),
    facecolor="white",
)

for ax, spec in zip(axes, plot_specs):
    counts_df = (
        spec["counts"]
        .head(10)
        .copy()
        .sort_values("count", ascending=True)
    )

    ax.barh(
        counts_df["window_label"],
        counts_df["count"],
    )

    ax.set_title(
        f'{spec["title"]}\n{spec["subtitle"]}',
        fontsize=12,
    )

    ax.set_xlabel("Number of selections")
    ax.set_ylabel("Window size")
    ax.grid(axis="x", alpha=0.3)

    for i, value in enumerate(counts_df["count"]):
        ax.text(
            value,
            i,
            f" {value}",
            va="center",
            fontsize=9,
        )

fig.suptitle(
    "Tuned benchmark - Top sliding-window values by Macro F1 ranking",
    fontsize=16,
    y=1.03,
)

fig.tight_layout()

output_file = (
    TOP_WINDOW_DIR
    / "tuned_top_window_values_top1_top3_top5_single_page.png"
)

fig.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

print("Saved:", output_file)
plt.show()

## Plot 3 — Good-enough region

A window is good enough if it reaches at least 95% of the best tuned Macro F1 within the same `dataset + classifier + stride_ratio`.

In [ ]:
GOOD_ENOUGH_DIR = PLOTS_ROOT / "good_enough_region"
GOOD_ENOUGH_DIR.mkdir(parents=True, exist_ok=True)
GOOD_ENOUGH_THRESHOLD = 0.95

good_df = ok_tuned_results.copy()
good_df["window_percentage_100"] = (good_df["window_percentage"] * 100).round(0).astype(int)
group_keys = ["dataset", "classifier", "stride_ratio"]

good_df["best_macro_f1_in_group"] = good_df.groupby(group_keys)["series_macro_f1"].transform("max")
good_df["relative_macro_f1"] = good_df["series_macro_f1"] / good_df["best_macro_f1_in_group"]
good_df["is_good_enough"] = good_df["relative_macro_f1"] >= GOOD_ENOUGH_THRESHOLD

good_region_by_stride = (
    good_df
    .groupby(["window_percentage_100", "stride_ratio"], as_index=False)
    .agg(good_enough_count=("is_good_enough", "sum"), total_count=("is_good_enough", "count"), good_enough_share=("is_good_enough", "mean"))
    .sort_values(["stride_ratio", "window_percentage_100"])
)

good_region_by_stride.to_csv(GOOD_ENOUGH_DIR / "tuned_good_enough_region_by_stride.csv", index=False)
display(good_region_by_stride.head())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), facecolor="white")

for stride_ratio, group in good_region_by_stride.groupby("stride_ratio"):
    group = group.sort_values("window_percentage_100")
    ax.plot(group["window_percentage_100"], group["good_enough_share"] * 100, marker="o", linewidth=2, label=f"stride ratio = {stride_ratio}")

ax.set_title(f"Tuned benchmark - Good-enough window region by stride ratio ({GOOD_ENOUGH_THRESHOLD:.0%} of best Macro F1)")
ax.set_xlabel("Window size (%)")
ax.set_ylabel("Good-enough configurations (%)")
ax.grid(True, alpha=0.3)
ax.legend(title="Stride ratio")
fig.tight_layout()

output_file = GOOD_ENOUGH_DIR / "tuned_good_enough_region_by_stride.png"
fig.savefig(output_file, dpi=300, bbox_inches="tight", facecolor="white")
print("Saved:", output_file)
plt.show()

## Plot 4 — Runtime summary

In [ ]:
RUNTIME_DIR = PLOTS_ROOT / "runtime"
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

runtime_summary = (
    ok_tuned_results
    .groupby("classifier", as_index=False)
    .agg(
        total_time_sec=("total_time_sec", "sum"),
        mean_total_time_sec=("total_time_sec", "mean"),
        median_total_time_sec=("total_time_sec", "median"),
        mean_fit_time_sec=("fit_time_sec", "mean"),
        mean_predict_time_sec=("predict_time_sec", "mean"),
        n_configs=("total_time_sec", "count"),
    )
    .sort_values("total_time_sec", ascending=False)
)

runtime_summary["total_time_min"] = runtime_summary["total_time_sec"] / 60
runtime_summary["total_time_hour"] = runtime_summary["total_time_sec"] / 3600
runtime_summary.to_csv(RUNTIME_DIR / "tuned_runtime_summary_by_classifier.csv", index=False)
display(runtime_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7), facecolor="white")
plot_df = runtime_summary.sort_values("total_time_min", ascending=True)
ax.barh(plot_df["classifier"], plot_df["total_time_min"])
ax.set_title("Tuned benchmark - Total final-run computation time by classifier")
ax.set_xlabel("Total final-run time (minutes)")
ax.set_ylabel("Classifier")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()

output_file = RUNTIME_DIR / "tuned_total_final_run_time_by_classifier.png"
fig.savefig(output_file, dpi=300, bbox_inches="tight", facecolor="white")
print("Saved:", output_file)
plt.show()

## Optional — Compare tuned and untuned results

Run this section only after the untuned benchmark has already produced:

```text
results/benchmark_10datasets_weasel_quant/untuned/combined_untuned_results.csv
```

In [ ]:
UNTUNED_COMBINED_FILE = Path("results/benchmark_10datasets_weasel_quant/untuned/combined_untuned_results.csv")

if UNTUNED_COMBINED_FILE.exists():
    untuned_results = pd.read_csv(UNTUNED_COMBINED_FILE)
    for column in numeric_columns:
        if column in untuned_results.columns:
            untuned_results[column] = pd.to_numeric(untuned_results[column], errors="coerce")
    untuned_ok = untuned_results[untuned_results["status"] == "ok"].copy()
    tuned_compare = ok_tuned_results.copy()
    untuned_compare = untuned_ok.copy()
    tuned_compare["run_type"] = "Tuned"
    untuned_compare["run_type"] = "Untuned"
    common_columns = sorted(set(tuned_compare.columns).intersection(set(untuned_compare.columns)))
    comparison_df = pd.concat([untuned_compare[common_columns], tuned_compare[common_columns]], ignore_index=True)
    comparison_file = TUNED_ROOT / "tuned_vs_untuned_combined_results.csv"
    comparison_df.to_csv(comparison_file, index=False)
    print("Saved:", comparison_file)
    display(comparison_df.head())
else:
    print("Untuned combined file not found:", UNTUNED_COMBINED_FILE)
    comparison_df = None

In [ ]:
if comparison_df is not None:
    comparison_summary = (
        comparison_df
        .groupby(["run_type", "window_percentage", "stride_ratio"], as_index=False)
        .agg(mean_macro_f1=("series_macro_f1", "mean"), n_configs=("series_macro_f1", "count"))
    )
    comparison_summary["window_percentage_100"] = (comparison_summary["window_percentage"] * 100).round(0).astype(int)

    stride_values = sorted(comparison_summary["stride_ratio"].dropna().unique())
    fig, axes = plt.subplots(1, len(stride_values), figsize=(6 * len(stride_values), 5), sharey=True, facecolor="white")
    if len(stride_values) == 1:
        axes = [axes]

    for ax, stride_ratio in zip(axes, stride_values):
        subset = comparison_summary[comparison_summary["stride_ratio"] == stride_ratio]
        for run_type, group in subset.groupby("run_type"):
            group = group.sort_values("window_percentage_100")
            ax.plot(group["window_percentage_100"], group["mean_macro_f1"], marker="o", linewidth=2, label=run_type)
        ax.set_title(f"Stride ratio = {stride_ratio}")
        ax.set_xlabel("Window size (%)")
        ax.grid(True, alpha=0.3)

    axes[0].set_ylabel("Mean series Macro F1")
    axes[-1].legend(title="Run type")
    fig.suptitle("Tuned vs Untuned - Mean Macro F1 by window size", y=1.03)
    fig.tight_layout()

    output_file = PLOTS_ROOT / "tuned_vs_untuned_mean_macro_f1_by_stride.png"
    fig.savefig(output_file, dpi=300, bbox_inches="tight", facecolor="white")
    print("Saved:", output_file)
    plt.show()
else:
    print("comparison_df is None.")

## Interpretation

Use this tuned benchmark as a robustness check, not as the main evidence for the direct window-size effect.

```text
Untuned: isolates the effect of window size.
Tuned: checks whether the same window regions remain competitive after model optimization.
```